# Injini — train & evaluate

Ships the DCASE Task 2 first-shot recipe compressed for an Arm phone: a frozen
AudioSet-distilled MobileNetV3 embedder plus a Mahalanobis distance score.
This notebook produces the evaluation table.

**Settings:** Internet ON. Add data: `zeyadzsm/engine-sounds`. Accelerator: GPU
optional. Then Run All. Artifacts land in `/kaggle/working`.

## 1. Environment

In [ ]:
!pip -q install onnx onnxruntime "onnxruntime-tools" hear21passt torchaudio 2>/dev/null
import torch, sys, os, json, numpy as np
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
os.makedirs("/kaggle/working/models", exist_ok=True)

## 2. Get the code and the EfficientAT weights

In [ ]:
!git clone -q https://github.com/rapha18th/injini.git || (cd injini && git pull -q)
%cd injini
!git clone -q https://github.com/fschmid56/EfficientAT.git vendor_efficientat || true
!mkdir -p vendor_efficientat/resources
for f in ["mn10_as_mAP_471.pt", "mn04_as_mAP_432.pt"]:
    p = f"vendor_efficientat/resources/{f}"
    if not os.path.exists(p):
        !wget -q -O {p} https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/{f}
!python src/features.py     # writes models/mel_kaldi_128x513.npy
sys.path.insert(0, "src")

## 3. DCASE 2025 Task 2 development set

In [ ]:
!python src/fetch_dcase.py --out data/dcase2025_dev
import glob
print(len(glob.glob("data/dcase2025_dev/*/*/*.wav")), "wav files")

## 4. Reproduce the DCASE autoencoder baseline

In [ ]:
!python src/baseline_ae.py --root data/dcase2025_dev --epochs 100

## 5. Reference ceiling — PaSST transformer embedding + Mahalanobis
The published state of the art in one reproducible form. This is the number the
phone pipeline is measured against.

In [ ]:
!python src/eval_dcase.py --root data/dcase2025_dev --backend passt --scorer maha     --out models/eval_passt_maha.json

## 6. Frozen EfficientAT mn10_as — FP32, then INT8

In [ ]:
!python src/embedder.py --name mn10_as --out models/injini_mn10_as_fp32.onnx
!python src/eval_dcase.py --root data/dcase2025_dev --backend onnx:models/injini_mn10_as_fp32.onnx     --scorer maha --out models/eval_mn10_fp32_maha.json
!python export/quantize.py --fp32 models/injini_mn10_as_fp32.onnx     --calib-dir data/dcase2025_dev --n-calib 256
!python src/eval_dcase.py --root data/dcase2025_dev --backend onnx:models/injini_mn10_as_int8.onnx     --scorer maha --out models/eval_mn10_int8_maha.json

## 7. Smaller candidate — mn04_as, FP32 and INT8

In [ ]:
!python src/embedder.py --name mn04_as --out models/injini_mn04_as_fp32.onnx
!python src/eval_dcase.py --root data/dcase2025_dev --backend onnx:models/injini_mn04_as_fp32.onnx     --scorer maha --out models/eval_mn04_fp32_maha.json
!python export/quantize.py --fp32 models/injini_mn04_as_fp32.onnx     --calib-dir data/dcase2025_dev --n-calib 256
!python src/eval_dcase.py --root data/dcase2025_dev --backend onnx:models/injini_mn04_as_int8.onnx     --scorer maha --out models/eval_mn04_int8_maha.json

## 8. Supervised fault-ID head (Kaggle engine-sounds, source-disjoint)

In [ ]:
ES = "/kaggle/input/engine-sounds"
!python src/prepare_engine_sounds.py --root {ES} --out data/engine_sounds_manifest.csv
!python src/faultid.py --manifest data/engine_sounds_manifest.csv --backend onnx:models/injini_mn10_as_fp32.onnx

## 9. Collect the table

In [ ]:
import glob, json
rows = []
for p in sorted(glob.glob("models/eval_*.json")) + sorted(glob.glob("models/faultid_*.json")):
    d = json.load(open(p))
    rows.append({"file": os.path.basename(p),
                 "official_score": d.get("official_score"),
                 "mean_auc": d.get("mean_auc"),
                 "macro_f1": d.get("macro_f1")})
summary = {"results": rows}
json.dump(summary, open("/kaggle/working/injini_metrics.json", "w"), indent=2)
for r in rows: print(r)
!cp -r models /kaggle/working/ 2>/dev/null
print("\nartifacts in /kaggle/working/models and /kaggle/working/injini_metrics.json")